# 01. Основы: имена, объекты, изменяемость

Неделя 1.

**Как работать с блокнотом**
- Сначала решаешь сам, без ИИ. В задачах «предскажи» сначала пишешь ответ в комментарий, потом запускаешь.
- Непонятное поведение — прогони фрагмент в [pythontutor.com](https://pythontutor.com/) и посмотри на стрелки «имя → объект».
- Ячейки Jupyter можно запускать в любом порядке, и состояние (переменные) копится между ними. Это источник «у меня работало». После всех выполненных заданий: **Kernel → Restart & Run All** — блокнот должен пройти сверху вниз без ошибок.

## Функция проверки `check`

Это **не чёрный ящик** — её тоже нужно понимать построчно.

- `cases` — список пар `(аргументы, ожидаемый результат)`. Аргументы — кортеж, поэтому для одного аргумента пишется `(x,)` с запятой: `(x)` — это просто `x` в скобках.
- `fn(*args)` распаковывает кортеж в позиционные аргументы: `fn(*(1, 2))` то же, что `fn(1, 2)`.
- `_same` сравнивает результаты. Вещественные числа сравниваются **приближённо** (`math.isclose`): `0.1 + 0.2 == 0.3` в Python — `False`, об этом ниже. Списки и кортежи сравниваются поэлементно и **с учётом типа**: если ожидается кортеж, список не подойдёт.
- `assert условие, сообщение` бросает `AssertionError` с сообщением, если условие ложно.
- `!r` в f-строке выводит `repr` значения: строки будут в кавычках, и `'1'` не спутать с `1`.

In [1]:
import math

def _same(a, b):
    if isinstance(a, float) or isinstance(b, float):
        if a is None or b is None:
            return a is b
        return math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b

def check(fn, cases):
    """cases: список пар (кортеж аргументов, ожидаемый результат)."""
    for args, expected in cases:
        got = fn(*args)
        assert _same(got, expected), f"{fn.__name__}{args!r}: ожидалось {expected!r}, получено {got!r}"
    print(f"{fn.__name__}: OK ({len(cases)} теста(ов))")

## 1. Имя — это не коробка, а ярлык

Главная модель Python, без которой дальше всё будет «магией»:

- Любое значение — это **объект**. У объекта есть *идентичность* (`id`), *тип* (`type`) и *значение*.
- Переменная — это **имя**, привязанное к объекту. Само имя ничего не хранит, оно указывает на объект.
- Оператор `=` **никогда не копирует** объект. Он привязывает имя слева к объекту справа. Это верно для любых типов: `int`, `str`, `list` — без исключений.

Проверить, один ли это объект, можно двумя способами:
- `a is b` — указывают ли имена на **один и тот же объект**;
- `id(a) == id(b)` — то же самое, через идентификаторы.

А `a == b` проверяет другое — **равны ли значения**: два разных объекта могут быть равны.

In [2]:
a = [1, 2, 3]
b = a                                 # второе имя для ТОГО ЖЕ списка, копии нет
b.append(4)                           # меняем объект через имя b
print(f"{a=}, {b=}")
print(f"{(a is b)=}, {(a == b)=}\n")  # изменение видно и через a: объект один

c = a.copy()                          # явно создаём НОВЫЙ объект с тем же содержимым
c.append(5)                           # меняем копию
print(f"{a=}, {c=}")
print(f"{(a is c)=}, {(a == c)=}")    # значения частично равны, но объекты разные

a=[1, 2, 3, 4], b=[1, 2, 3, 4]
(a is b)=True, (a == b)=True

a=[1, 2, 3, 4], c=[1, 2, 3, 4, 5]
(a is c)=False, (a == c)=False


### 1.1. Перепривязка или изменение объекта

С объектом можно сделать две принципиально разные вещи:

| Операция | Что происходит | Пример |
|---|---|---|
| **Перепривязка имени** | Имя начинает указывать на *другой* объект. Сам старый объект не меняется, другие имена его по-прежнему видят. | `a = 0`, `a = [0]`, `a = a + 1`  |
| **Изменение объекта** | Меняется *содержимое* объекта. Это видно через **все** имена, которые на него указывают. | `a.append(3)`, `a[0] = 7`, `d["k"] = 1` |

Частое заблуждение: «числа копируются, а списки передаются по ссылке». Это неверно. Числа тоже не копируются — просто у `int` **нет операций изменения**, с ним можно делать только перепривязку.

In [3]:
a = 1000
b = a
print(f"Один объект?", {a is b}, {id(a) == id(b)})  # True True — копии нет даже для int

a = 0                # перепривязка: имя a теперь указывает на объект 0
print(a, b, a is b)  # b по-прежнему указывает на 1000

# со списком — ровно то же самое, если делать перепривязку
x = [1, 2]
y = x
x = [0]      # перепривязка, а не изменение
print(x, y)  # y не изменился

Один объект? {True} {True}
0 1000 False
[0] [1, 2]


### 1.2. Изменяемые и неизменяемые типы

| Неизменяемые (immutable) | Изменяемые (mutable) |
|---|---|
| `int`, `float`, `complex`,<br>`bool`, `str`, `bytes`,<br>`tuple`, `frozenset`, `None` | `list`, `dict`, `set`, `bytearray`,<br>большинство объектов<br>пользовательских классов |

У неизменяемого объекта нельзя поменять значение «на месте». Поэтому любая «модификация» создаёт **новый** объект:
- `s.upper()` у строки возвращает новую строку, исходная не меняется;
- `x + 1` у числа даёт новый объект-число.

### 1.3. Ловушка `+=`

`a += b` выглядит одинаково для всех типов, но работает по-разному:
- если у типа есть изменение на месте (метод `__iadd__`, как у `list`), объект **меняется**, имя остаётся привязанным к нему же;
- если нет (как у `int`, `str`, `tuple`), выполняется `a = a + b` — создаётся **новый** объект и имя **перепривязывается**.

In [4]:
n = 0
m = n
n += 1  # для int: n = n + 1 -> новый объект, перепривязка
print(f"{n=}, {m=}, {(n is m)=}\n")  # 1 0 False

lst = [0]
alias = lst
lst += [1]  # для list: изменение на месте (то же, что lst.extend([1]))
print(f"{lst=}, {alias=}, {(lst is alias)=} \n")  # [0, 1] [0, 1] True

lst = lst + [2]  # а вот это — создание нового списка и перепривязка!
print(f"{lst=}, {alias=}, {(lst is alias)=}")  # [0, 1, 2] [0, 1] False

n=1, m=0, (n is m)=False

lst=[0, 1], alias=[0, 1], (lst is alias)=True 

lst=[0, 1, 2], alias=[0, 1], (lst is alias)=False


### 1.4. `is` — только для `None`

`is` сравнивает идентичность, а не значение. Для сравнения чисел и строк он **не подходит**: результат зависит от деталей реализации. CPython заранее создаёт и переиспользует объекты для целых от −5 до 256, поэтому для маленьких чисел `is` «случайно» работает, а для больших — нет.

Правило: `x is None` / `x is not None` — правильно; `x == 5`, `s == "abc"` — правильно; `x is 5` — ошибка (Python даже выдаст `SyntaxWarning`).

In [5]:
small_a, small_b = int("256"), int("256")  # int(...) — чтобы числа создались во время выполнения
big_a, big_b = int("257"), int("257")
print(f"{(small_a is small_b)=}")  # True  — объект из кэша маленьких чисел
print(f"{(big_a is big_b)=}")      # False — два разных объекта
print(f"{(big_a == big_b)=}")      # True  — значения равны: для чисел нужен именно ==

value = None
print(f"{(value is None)=}")  # правильная проверка на None

(small_a is small_b)=True
(big_a is big_b)=False
(big_a == big_b)=True
(value is None)=True


### 1.5. Поверхностная и глубокая копия

`list.copy()`, `list(a)` и срез `a[:]` создают **поверхностную** копию: новый внешний список, но элементы — те же самые объекты. Если элементы изменяемые (например, вложенные списки), они оказываются общими.

`copy.deepcopy(a)` рекурсивно копирует и вложенные объекты.

In [6]:
import copy

m = [[1, 2], [3, 4]]
shallow = m.copy()
deep = copy.deepcopy(m)

m[0].append(99)       # меняем ВЛОЖЕННЫЙ список
print(f"{shallow=}")  # [[1, 2, 99], [3, 4]] — вложенные списки общие
print(f"{deep=}")     # [[1, 2], [3, 4]]     — полностью независимая копия
print(f"{(m[0] is shallow[0])=}, {(m[0] is deep[0])=}")

shallow=[[1, 2, 99], [3, 4]]
deep=[[1, 2], [3, 4]]
(m[0] is shallow[0])=True, (m[0] is deep[0])=False


### Задача 1. Предскажи вывод

Запиши ответ в комментарий **до** запуска. После запуска объясни каждую строку в терминах «имя → объект»: где перепривязка, где изменение объекта, какие объекты общие. Если прогноз не совпал — прогони код в [pythontutor](https://pythontutor.com/).

In [7]:
# Мой прогнозы в комментариях AS:
# AS: 11, 10
x = 10
y = x
x += 1
print(x, y)

# AS: [[7, 0], [7, 0], [7, 0]]
s = [[0] * 2] * 3
s[0][0] = 7
print(s)
# Операция [[0] * 2] * 3 не создает 3 отдельных вложенных
# списка. Она создает один список [0, 0] и дублирует ссылку
# на него 3 раза. А так как list — изменяемый тип, изменение
# элемента внутри одного вложенного списка модифицирует общий
# объект, и изменения отображаются во всех трёх.


# AS: (1, [2, 3, 4])
t = (1, [2, 3])
t[1].append(4)
print(t)

11 10
[[7, 0], [7, 0], [7, 0]]
(1, [2, 3, 4])


## 2. Как аргументы попадают в функцию

В Python нет выбора «по значению / по ссылке». Параметр функции — это **новое локальное имя**, привязанное к тому же объекту, что передал вызывающий. Это называют *передачей через присваивание* (call by sharing).

Отсюда два следствия:
- если функция **изменяет** объект (`xs.append(...)`), изменение видно снаружи;
- если функция **перепривязывает** параметр (`xs = [...]`), это влияет только на локальное имя, снаружи ничего не меняется.

In [8]:
def modify(xs):
    xs.append(1)  # изменение объекта: видно снаружи
    xs = [99]     # перепривязка локального имени: снаружи не видно
    xs.append(2)  # меняем уже НОВЫЙ локальный список

data = [0]
modify(data)
print(data)       # [0, 1]

[0, 1]


### Задача 2. Ловушка изменяемого аргумента по умолчанию

Функция ниже содержит классическую ошибку.
1. Запусти и объясни, **в какой момент** создаётся список `[]` из значения по умолчанию: при каждом вызове или один раз? Подсказка: посмотри `add_item.__defaults__` после каждого вызова.
2. Исправь функцию. Стандартный приём: значение по умолчанию `None`, а новый список создаётся внутри функции, если аргумент не передан.
3. Почему такой проблемы нет у `def f(x=0)` и `def f(s="")`?

In [9]:
def add_item(item, items=[]):
    items.append(item)
    return items

print(add_item(1))
print(add_item(2))  # почему здесь не [2]?

# Исправленная версия
def add_item(item, items: list | None = None):
    if items is None:
        items: list = []
    items.append(item)
    return items

print(add_item(1))
print(add_item(2))

[1]
[1, 2]
[1]
[2]


Нашёл неплохую [статейку](https://habr.com/en/articles/982534/) на Хабре по этой теме

## 3. Практика: функции без готовых решений

Нельзя использовать `sum`, `max`, `min`, `sorted`, `statistics`, `collections` — только циклы, условия и базовые операции. Смысл — понять, что эти функции делают внутри.

### Задача 3. Среднее и дисперсия

`mean_var(xs)` возвращает **кортеж** `(среднее, дисперсия)`, где дисперсия считается с делением на `n`:

$$\bar x = \frac{1}{n}\sum x_i, \qquad D = \frac{1}{n}\sum (x_i - \bar x)^2.$$

Для пустого списка — `(None, None)`.

**Тонкость про `float`.** Вещественные числа хранятся в двоичном виде, и большинство десятичных дробей (`0.1`, `0.2`) представлены неточно. Поэтому вещественные результаты сравнивают с допуском (`math.isclose`), а не через `==`. Запусти ячейку ниже, прежде чем решать.

**Вопрос:** чем дисперсия с делением на `n` отличается от *выборочной исправленной* с делением на `n − 1` и когда нужна какая?

**Ответ:** с делением на `n` даёт смещенную оценку дисперсии **для выборки**, а с делением на `n - 1` — несмещенную оценку дисперсии **для генеральной совокупности** благодаря [поправке Бесселя](https://practicum.yandex.ru/blog/dispersiya-v-statistike/).

При этом и та, и та вычисляются на одной и той же выборке.

In [10]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3))
print(f"{0.1:.20f}")   # как 0.1 хранится на самом деле

0.30000000000000004
False
True
0.10000000000000000555


In [11]:
def mean_var(xs):
    if len(xs) == 0:
        return None, None
    
    value_sum = 0
    for value in xs:
        value_sum += value
    mean = value_sum / len(xs)

    var_sum = 0
    for value in xs:
        var_sum += (value - mean)**2
    var = var_sum / len(xs)

    return mean, var

check(mean_var, [
    (([1, 2, 3, 4],), (2.5, 1.25)),
    (([5],), (5.0, 0.0)),
    (([0.1, 0.2, 0.3],), (0.2, 0.02 / 3)),
    (([],), (None, None)),
])

mean_var: OK (4 теста(ов))


### Задача 4. Частоты слов

`word_freq(text)` → словарь `{слово: количество}`. Регистр не важен, знаки `. , ! ?` игнорируются.

**Подсказки и тонкости:**
- строка неизменяема: `text.replace(".", " ")` и `text.lower()` возвращают **новые** строки, результат нужно присвоить;
- `text.split()` без аргументов делит по любым пробельным символам и выбрасывает пустые куски, а `text.split(" ")` — нет (проверь на `"a  b"`);
- `d.get(key, 0)` возвращает значение или `0`, если ключа нет; `d[key]` для отсутствующего ключа бросит `KeyError`;
- словари сравниваются по содержимому без учёта порядка ключей.

In [12]:
def word_freq(text):
    if len(text) == 0:
        return {}
    
    for c in ['.', ',', '!', '?']:
        text = text.replace(c, ' ');
    text = text.lower()
    text_list = text.strip().split()

    freq_count: dict = {}
    for i, word in enumerate(text_list):
        freq_count.setdefault(word, 1);
        for j in range(i+1, len(text_list)):
            if text_list[j] == word:
                freq_count[word] += 1;

    return freq_count

check(word_freq, [
    (("Кот и кот. И пёс!",), {"кот": 2, "и": 2, "пёс": 1}),
    (("a  b,a",), {"a": 2, "b": 1}),
    (("",), {}),
])

word_freq: OK (3 теста(ов))


### Задача 5. Бинарный поиск

`bsearch(xs, target)` — индекс `target` в **отсортированном** списке или `-1`. Сложность O(log n).

**Что нужно объяснить:**
- *инвариант цикла*: что гарантированно верно про границы `lo` и `hi` на каждой итерации;
- почему цикл гарантированно завершится (что происходит с длиной отрезка `[lo, hi]`);
- в Python `(lo + hi) // 2` безопасно, потому что `int` не переполняется (длинная арифметика). В C++ это классическая ошибка переполнения, там пишут `lo + (hi - lo) / 2` — вернёмся к этому в месяце 1 по C++;
- `//` — целочисленное деление с округлением **вниз**: `7 // 2 == 3`, но `-7 // 2 == -4`.

In [13]:
def bsearch(xs, target):
    if len(xs) == 0:
        return -1

    lo = 0
    hi = len(xs) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if xs[mid] == target:
            return mid
        elif xs[mid] > target:
            hi = mid - 1
        elif xs[mid] < target:
            lo = mid + 1
    return -1

data = [1, 3, 5, 7, 9, 11]
check(bsearch, [((data, 7), 3), ((data, 1), 0), ((data, 11), 5),
                ((data, 4), -1), ((data, 0), -1), ((data, 12), -1), (([], 1), -1)])

bsearch: OK (7 теста(ов))


### Задача 6. Транспонирование матрицы

`transpose(m)` — транспонирование списка списков **без** `zip` и `numpy`. Исходная матрица не должна измениться, а результат не должен делить с ней ни одного вложенного списка.

Для пустой матрицы `[]` результат — `[]`. Считаем, что все строки одной длины.

Подумай, почему решение «скопировать `m` и поменять элементы местами» здесь не работает для неквадратной матрицы и чем опасна поверхностная копия.

In [14]:
def transpose(m):
    if len(m) == 0:
        return []
    
    t = []
    for j in range(0, len(m[0])):
        for i in range(0, len(m)):
            if i == 0:
                t.append([m[i][j]])
            else:
                t[j].append(m[i][j])
    return t

m = [[1, 2, 3],
     [4, 5, 6]]
check(transpose, [((m,), [[1, 4], [2, 5], [3, 6]]), (([[7]],), [[7]]), (([],), [])])
assert m == [[1, 2, 3],
             [4, 5, 6]], "исходная матрица изменилась"
t = transpose(m)
assert all(row is not orig for row in t for orig in m), "результат делит вложенные списки с исходной матрицей"
print("исходник цел, общих вложенных списков нет")

transpose: OK (3 теста(ов))
исходник цел, общих вложенных списков нет


## Вопросы
1. Чем `is` отличается от `==`? Почему `x is 257` — ошибка, а `x is None` — нет?
2. Чем перепривязка имени отличается от изменения объекта? Приведи по два примера каждого.
3. Какие типы изменяемые, какие нет? Почему кортеж, внутри которого есть список, «можно изменить»?
4. Почему `[[0] * 2] * 3` — ловушка? Как правильно создать матрицу 3×2 из нулей?
5. Почему `n += 1` для `int` и `lst += [1]` для списка ведут себя по-разному?
6. Чем поверхностная копия отличается от глубокой? Когда хватает `a.copy()`?
7. Что произойдёт с аргументом снаружи, если внутри функции сделать `xs = []`, а что — если `xs.append(1)`?
8. Когда вычисляется значение аргумента по умолчанию?

## Ответы
1. `is` сравнивает идентификаторы объектов, а не значения.`x is 257` не вызовет ошибку как таковую, однако вернёт `False` из-за особенностей управления памятью в CPython — при запуске каждого нового процесса (интерпретатора) в куче (**heap**) выделяется массив из 262 структур PyLongObject - это и есть числа от -5 до 256, они создаются один раз и живут всё время, пока работает процесс. `None` в Python это специальный объект **singleton класса**, а как известно объект такого класса в отдельном процессе может быть только один, поэтому и `id(None)` тоже всегда будет возвращать один и тот же `int`.
2. Перепривязка (**rebinding**) просто присваивает той же самой ссылке новый объект, в отличие от изменения объекта при котором изменяются содержимое и/или состояние объекта. Примеры перепривязки: `l = [1, 2, 3]; l = [1, 2, 3, 4]` и `num = 0; num += 1`. Примеры изменения: `l = [1, 2, 3]; l.append(4)` и `d = {'a': 1, 'b': 2, 'c': 3}; d.setdefault('d', 4)`.
3. Выше в блокноте таблица **immutable** и **mutable** типов, см. её. Потому что кортеж неизменяем в том смысле, что его структуру (набор ссылок) поменять нельзя. Однако если внутри кортежа лежит изменяемый объект, то меняется внутреннее состояние самого объекта, на который ссылается элемент кортежа, сама ссылка (адрес в памяти) в кортеже при этом остаётся прежней
4. Операция `[[0] * 2] * 3` не создает 3 отдельных вложенных списка. Она создает один список `[0, 0]` и дублирует ссылку на него 3 раза. А так как list — изменяемый тип, изменение элемента внутри одного вложенного списка модифицирует общий объект, и изменения отображаются во всех трёх. Правильно было бы сделать `[[0, 0] for i in range(0, 3)]`.
5. Потому что `n += 1` просто перепривязывает имени `n` новый объект, а `lst += [1]` изменяет объект `lst` благодаря методу `__iadd__`.
6. Поверхностная копия создаёт только новый объект-контейнер и заполняет его ссылками на те же элементы в памяти, а глубокая копирует всё рекурсивно. Поверхностной копии хватает, когда объект содержит только неизменяемые поля или когда не требуется изменять поля этого объекта.
7. Если внутри функции сделать `xs = []` это будет означать что локальному имени `xs` будет присвоен новый объект — пустой список и объект снаружи функции не пострадает. Если сделать `xs.append(1)` объект изменится и снаружи функции, так происходит потому, что в Python аргументы передаются по ссылке на объект (**pass-by-sharing**): функция получает копию ссылки на тот же самый список в памяти, что позволяет мутировать его содержимое, но не позволяет переприсвоить саму внешнюю переменную.
8. Ровно один раз — в момент определения функции.